<a href="https://colab.research.google.com/github/zkhadija132/flyrank-ml-assignment/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train"
)

df = pd.DataFrame(ds.take(1000))

df.head()

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
print(f"Dataset Shape: {df.shape}")

df['report_date'] = pd.to_datetime(df['report_date'])

min_date = df['report_date'].min()
max_date = df['report_date'].max()
total_days = (max_date - min_date).days + 1

print(f"Time Window: {min_date.strftime('%Y-%m-%d')} to {max_date.strftime('%Y-%m-%d')} ({total_days} days)")

Dataset Shape: (1000, 30)
Time Window: 2025-01-27 to 2025-01-30 (4 days)


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Unit of analysis:

One row represents daily performance of one content item for one pseudonymized client on a specific report date.

Features:

gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users, engagement metrics, traffic channel sessions, and AI referral metrics.

Label:

The dataset does not provide a direct prediction label. Performance outcomes such as clicks, sessions, or pageviews can be treated as observed metrics depending on the decision question.

Context:

Client availability flags (GSC/GA4), report date, client hash ID, content hash ID, and data availability indicators.

Excluded:

Client and content hash IDs are excluded as predictive features because they are identifiers, not meaningful behavioral signals. Private client information is not used.

In [ ]:
df.columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events']

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# Row count
print("Number of rows:", len(df))

# Check duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Missing values
print("\nMissing values:")
print(df.isnull().sum())

# Date window verification
print("\nDate range:")
print(df['report_date'].min(), "to", df['report_date'].max())

# Unique clients and content items
print("\nUnique clients:", df['client_hash_id'].nunique())
print("Unique content items:", df['content_hash_id'].nunique())

Number of rows: 1000
Duplicate rows: 0

Missing values:
report_date                 0
client_hash_id              0
content_hash_id             0
client_has_gsc              0
client_has_ga4              0
gsc_data_available          0
ga4_data_available          0
gsc_impressions             0
gsc_clicks                  0
gsc_sum_position            0
gsc_avg_position            0
ga4_pageviews               0
ga4_sessions                0
ga4_users                   0
ga4_engaged_sessions        0
ga4_total_engagement_sec    0
sessions_organic            0
sessions_direct             0
sessions_referral           0
sessions_social             0
sessions_paid               0
sessions_ai                 0
ai_chatgpt                  0
ai_perplexity               0
ai_gemini                   0
ai_copilot                  0
ai_claude                   0
ai_meta                     0
ai_other                    0
scroll_events               0
dtype: int64

Date range:
2025-01-27 00:00:0

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset has some limitations. The observed history is unbalanced because each client may have different available data periods. Early rows may only contain Google Search Console (GSC) data depending on client setup. The dataset shows associations and directional patterns, but it cannot prove causation. Overlapping time windows and missing external factors may affect interpretation. Client and content identifiers are pseudonymized, so individual entities cannot be identified.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.